# exp_7 — Experiment 7: SimpleRNN and GRU for Structured Time-Series Data

Source-based notebook converted from the uploaded experiment code document.


**Note:** The source uses the Delhi temperature CSV at a Windows path. Update that path if your dataset is stored elsewhere.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler

import tensorflow as tf
from tensorflow.keras.utils import timeseries_dataset_from_array
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, GRU, Dense

from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error

import warnings
warnings.filterwarnings("ignore")

In [ ]:
df = pd.read_csv(r"D:\DL DATASET\city_temp\city_temperature.csv")

print(df.shape)

df.head()

In [ ]:
print(df.info())

print("\nColumns\n")

print(df.columns)

print("\nMissing Values\n")

print(df.isnull().sum())

In [ ]:
print("Total Cities :", df["City"].nunique())

print(df["City"].unique())

In [ ]:
print(df.columns)

print(df.info())

print(df.isnull().sum())

In [ ]:
print(df["City"].nunique())

print(df["City"].unique()[:100])

In [ ]:
city_df = df[df["City"]=="Delhi"].copy()

city_df = city_df[city_df["AvgTemperature"]!=-99]

city_df.reset_index(drop=True,inplace=True)

print(city_df.shape)

city_df.head()

In [ ]:
city_df = city_df[city_df["AvgTemperature"] != -99]

city_df.reset_index(drop=True, inplace=True)

print(city_df.shape)

In [ ]:
city_df["Date"] = pd.to_datetime(

    city_df[["Year","Month","Day"]]

)

city_df = city_df.sort_values("Date")

city_df.reset_index(drop=True,inplace=True)

city_df.head()

In [ ]:
temperature = city_df[["AvgTemperature"]]

print(temperature.shape)

temperature.head()

In [ ]:
plt.figure(figsize=(15,5))

plt.plot(city_df["Date"],temperature)

plt.title("Delhi Daily Average Temperature")

plt.xlabel("Date")

plt.ylabel("Temperature")

plt.grid(True)

plt.show()

In [ ]:
scaler = MinMaxScaler()

scaled_temperature = scaler.fit_transform(temperature)

print(scaled_temperature.shape)

In [ ]:
train_size = int(len(scaled_temperature)*0.80)

train = scaled_temperature[:train_size]

test = scaled_temperature[train_size:]

print("Training Samples :",train.shape)

print("Testing Samples :",test.shape)

In [ ]:
from tensorflow.keras.utils import timeseries_dataset_from_array

lookback = 30
batch_size = 32

train_dataset = timeseries_dataset_from_array(
    data=train[:-lookback],
    targets=train[lookback:],
    sequence_length=lookback,
    sequence_stride=1,
    shuffle=False,
    batch_size=batch_size
)

test_dataset = timeseries_dataset_from_array(
    data=test[:-lookback],
    targets=test[lookback:],
    sequence_length=lookback,
    sequence_stride=1,
    shuffle=False,
    batch_size=batch_size
)

print("Training batches :", len(train_dataset))
print("Testing batches :", len(test_dataset))

In [ ]:
for x, y in train_dataset.take(1):

    print("Input Shape :", x.shape)

    print("Output Shape :", y.shape)

In [ ]:
from keras.utils import pad_sequences

sample_sequences = [

    [1,2,3],

    [4,5],

    [6,7,8,9]

]

padded = pad_sequences(

    sample_sequences,

    maxlen=5,

    padding="pre"

)

print("Original Sequences")

print(sample_sequences)

print()

print("Padded Sequences")

print(padded)

In [ ]:
simple_rnn = tf.keras.Sequential([

    tf.keras.layers.Input(shape=(lookback,1)),

    tf.keras.layers.SimpleRNN(
        64,
        activation="tanh"
    ),

    tf.keras.layers.Dense(
        32,
        activation="relu"
    ),

    tf.keras.layers.Dense(1)

])

simple_rnn.summary()

In [ ]:
simple_rnn.compile(

    optimizer="adam",

    loss="mse",

    metrics=["mae"]

)

In [ ]:
callbacks = [

    tf.keras.callbacks.EarlyStopping(

        monitor="val_loss",

        patience=5,

        restore_best_weights=True

    ),

    tf.keras.callbacks.ReduceLROnPlateau(

        monitor="val_loss",

        factor=0.5,

        patience=3,

        verbose=1

    )

]

In [ ]:
history_rnn = simple_rnn.fit(

    train_dataset,

    validation_data=test_dataset,

    epochs=10,

    callbacks=callbacks

)

In [ ]:
loss_rnn, mae_rnn = simple_rnn.evaluate(

    test_dataset,

    verbose=0

)

print("SimpleRNN Loss :", loss_rnn)

print("SimpleRNN MAE :", mae_rnn)

In [ ]:
stateful_rnn = tf.keras.Sequential([

    tf.keras.layers.Input(batch_shape=(1, lookback, 1)),

    tf.keras.layers.SimpleRNN(

        32,

        stateful=True

    ),

    tf.keras.layers.Dense(1)

])

stateful_rnn.compile(

    optimizer="adam",

    loss="mse"

)

print("Stateful SimpleRNN Created Successfully")

stateful_rnn.summary()

In [ ]:
gru_model = tf.keras.Sequential([

    tf.keras.layers.Input(shape=(lookback,1)),

    tf.keras.layers.GRU(
        64,
        activation="tanh"
    ),

    tf.keras.layers.Dense(
        32,
        activation="relu"
    ),

    tf.keras.layers.Dense(1)

])

gru_model.summary()

In [ ]:
gru_model.compile(

    optimizer="adam",

    loss="mse",

    metrics=["mae"]

)

In [ ]:
history_gru = gru_model.fit(

    train_dataset,

    validation_data=test_dataset,

    epochs=10,

    callbacks=callbacks

)

In [ ]:
loss_gru, mae_gru = gru_model.evaluate(

    test_dataset,

    verbose=0

)

print("GRU Loss :", loss_gru)

print("GRU MAE :", mae_gru)

In [ ]:
pred_rnn = simple_rnn.predict(test_dataset)

pred_gru = gru_model.predict(test_dataset)

actual = []

for _, y in test_dataset:

    actual.extend(y.numpy())

actual = np.array(actual)

In [ ]:
pred_rnn = scaler.inverse_transform(pred_rnn)

pred_gru = scaler.inverse_transform(pred_gru)

actual = scaler.inverse_transform(actual.reshape(-1,1))

In [ ]:
plt.figure(figsize=(15,6))

plt.plot(actual,label="Actual Temperature")

plt.plot(pred_rnn,label="SimpleRNN Prediction")

plt.plot(pred_gru,label="GRU Prediction")

plt.title("Actual vs Predicted Temperature")

plt.xlabel("Days")

plt.ylabel("Temperature")

plt.legend()

plt.grid(True)

plt.show()

In [ ]:
plt.figure(figsize=(12,5))

plt.plot(history_rnn.history["loss"],label="SimpleRNN Train")

plt.plot(history_rnn.history["val_loss"],label="SimpleRNN Validation")

plt.plot(history_gru.history["loss"],label="GRU Train")

plt.plot(history_gru.history["val_loss"],label="GRU Validation")

plt.title("Training and Validation Loss")

plt.xlabel("Epoch")

plt.ylabel("Loss")

plt.legend()

plt.grid(True)

plt.show()

In [ ]:
comparison = pd.DataFrame({

    "Model":["SimpleRNN","GRU"],

    "Loss":[loss_rnn,loss_gru],

    "MAE":[mae_rnn,mae_gru]

})

print(comparison)